# Entraînement Colab — feu rouge

La machine locale n'a pas de GPU (`torch 2.11.0+cpu`, `torch.cuda.is_available()` est faux). Ce notebook entraîne les trois détecteurs **de zéro**.

Aucun poids pré-entraîné : le modèle est `yolo11n.yaml` (ou `yolov8n.yaml`) et `pretrained=False`. Ne pas charger `yolov8n.pt`, `yolo11n.pt` ni `yolov4-obj.weights`.

Avant de lancer les cellules d'entraînement, zipper `data/processed` sur la machine locale et l'envoyer ici (fichier ou Google Drive).

In [ ]:
import torch
print(torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    raise SystemExit("Pas de GPU : ne pas lancer l'entraînement sur CPU.")

In [ ]:
%pip install -q ultralytics
from pathlib import Path
zip_path = Path("/content/processed.zip")
dest = Path("/content/data/processed")
if zip_path.is_file() and not dest.exists():
    import zipfile
    zipfile.ZipFile(zip_path).extractall("/content/data")
print("jeux", sorted(p.name for p in dest.iterdir()) if dest.is_dir() else "processed introuvable")

## Entraînement

Trois tâches, dans l'ordre : vehicles, traffic_lights, plates_detection.

- images 640, 100 époques, arrêt anticipé `patience=20`, graine 42
- augmentations par défaut d'Ultralytics
- véhicules : les quatre classes ont des effectifs proches (environ 500 à 700 boîtes). Pas de rééchantillonnage.
- Les résultats d'Ultralytics restent dans `/content/runs`.

In [ ]:
from ultralytics import YOLO

TASKS = ("vehicles", "traffic_lights", "plates_detection")
MODEL_YAML = "yolo11n.yaml"  # alternative : yolov8n.yaml

def train_task(task):
    data = f"/content/data/processed/{task}/data.yaml"
    model = YOLO(MODEL_YAML)
    return model.train(
        data=data,
        epochs=100,
        imgsz=640,
        patience=20,
        seed=42,
        pretrained=False,
        project="/content/runs",
        name=task,
        exist_ok=True,
    )

for task in TASKS:
    print("entraînement", task)
    train_task(task)